In [22]:
import pandas as pd
import numpy as np

In [23]:
np.random.seed(42)

In [ ]:
routes = ['BCN-LHR', 'MAD-CDG', 'MAD-LHR']
device = ['mobile', 'desktop']
user_id = [i for i in range(1,11)]
dates_range = pd.date_range('2023-01-01','2023-12-31')

n = 1000

routes_lst = np.random.choice(routes,size=n)
device_lst = np.random.choice(device,size=n)
user_id_lst = np.random.choice(user_id,size=n)
dates_range_lst = np.random.choice(dates_range,size=n)

searches = np.linspace(150,700, 1000) + np.random.randint(100,300,1000)
revenue = np.linspace(0,300, 1000) + np.random.randint(50,200,1000)

# np.random.choice(searches)
fake_data = pd.DataFrame({'user_id':user_id_lst, 'route':routes_lst,
                           'date':dates_range_lst,
                           'searches': searches,
                           'device':device_lst,
                             'revenue': revenue})

# fake_data

null_searches = np.random.choice(len(searches), 100, replace=False)
null_revenue = np.random.choice(len(revenue), 100, replace=False)

fake_data.loc[null_searches, 'searches'] = None
fake_data.loc[null_revenue, 'revenue'] = None

In [25]:
fake_data['searches'] = fake_data.groupby('route')['searches'].transform( lambda x: x.fillna(x.median()))

fake_data['revenue_imputed'] = fake_data['revenue'].isna().astype(int)
fake_data['revenue'] = fake_data.groupby('route')['revenue'].transform(
    lambda x: x.fillna(x.median())
)

In [44]:
fake_data[fake_data['revenue_imputed']==1]

,user_id,route,date,searches,device,revenue,revenue_imputed
28,2,MAD-LHR,2023-11-09,383.415415,desktop,256.797297,1
30,3,MAD-CDG,2023-11-22,281.516517,mobile,282.912913,1
32,7,MAD-CDG,2023-07-01,370.617618,desktop,282.912913,1
49,4,MAD-CDG,2023-09-27,332.976977,desktop,282.912913,1
63,10,MAD-CDG,2023-03-25,342.684685,mobile,282.912913,1
...,...,...,...,...,...,...,...
943,5,MAD-LHR,2023-10-25,876.169169,mobile,256.797297,1
950,7,MAD-LHR,2023-10-30,966.023023,desktop,256.797297,1
975,4,BCN-LHR,2023-10-19,637.111612,mobile,280.019520,1
978,7,MAD-CDG,2023-01-28,864.438438,desktop,282.912913,1


In [45]:
fake_data

,user_id,route,date,searches,device,revenue,revenue_imputed
0,1,MAD-LHR,2023-04-17,295.000000,mobile,140.000000,0
1,8,BCN-LHR,2023-04-23,637.111612,mobile,179.300300,0
2,2,MAD-LHR,2023-12-16,415.101101,mobile,111.600601,0
3,8,MAD-LHR,2023-09-11,352.651652,mobile,98.900901,0
4,3,BCN-LHR,2023-04-28,637.111612,desktop,187.201201,0
...,...,...,...,...,...,...,...
995,9,MAD-CDG,2023-02-22,912.797798,mobile,459.798799,0
996,4,MAD-CDG,2023-02-13,620.787788,desktop,479.099099,0
997,10,MAD-LHR,2023-10-30,919.898899,mobile,456.399399,0
998,2,MAD-LHR,2023-02-22,905.449449,mobile,457.699700,0


In [54]:
# fake_dup = fake_data.groupby(['user_id','route','date','device']).count().reset_index()

fake_data.drop_duplicates(subset=['user_id','route','date','device'], inplace= True)

In [55]:
# dups = fake_dup[fake_dup['searches']>1].index
# dups
fake_data

,user_id,route,date,searches,device,revenue,revenue_imputed
0,1,MAD-LHR,2023-04-17,295.000000,mobile,140.000000,0
1,8,BCN-LHR,2023-04-23,637.111612,mobile,179.300300,0
2,2,MAD-LHR,2023-12-16,415.101101,mobile,111.600601,0
3,8,MAD-LHR,2023-09-11,352.651652,mobile,98.900901,0
4,3,BCN-LHR,2023-04-28,637.111612,desktop,187.201201,0
...,...,...,...,...,...,...,...
995,9,MAD-CDG,2023-02-22,912.797798,mobile,459.798799,0
996,4,MAD-CDG,2023-02-13,620.787788,desktop,479.099099,0
997,10,MAD-LHR,2023-10-30,919.898899,mobile,456.399399,0
998,2,MAD-LHR,2023-02-22,905.449449,mobile,457.699700,0


In [53]:
# fake_data_no_dups = fake_data.iloc[~dups,]
# fake_data_no_dups

In [ ]:
# fake_data_no_dups.describe()

,user_id,date,searches,revenue,revenue_imputed
count,26.000000,26,26.000000,26.000000,26.000000
mean,5.769231,2023-06-24 07:23:04.615384576,644.671903,297.541522,0.076923
min,1.000000,2023-01-12 00:00:00,397.009009,116.459459,0.000000
25%,4.000000,2023-05-05 18:00:00,573.832332,280.750000,0.000000
50%,5.000000,2023-06-21 00:00:00,632.610110,295.696697,0.000000
75%,8.000000,2023-09-09 12:00:00,737.397147,345.505255,0.000000
max,10.000000,2023-12-15 00:00:00,905.888889,438.141141,1.000000
std,2.984060,NaN,121.483741,79.076245,0.271746


In [65]:
fake_data_agg = fake_data.groupby(['route', 'date']).agg(
    total_searches=('searches', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_revenue_per_search=('revenue', lambda x: x.sum() / fake_data.loc[x.index, 'searches'].sum())
).reset_index()

fake_data_agg

,route,date,total_searches,total_revenue,avg_revenue_per_search
0,BCN-LHR,2023-01-01,1402.945445,540.803303,0.385477
1,BCN-LHR,2023-01-05,1334.598098,636.516517,0.476935
2,BCN-LHR,2023-01-07,594.173173,239.276276,0.402705
3,BCN-LHR,2023-01-08,1458.190190,648.921922,0.445019
4,BCN-LHR,2023-01-10,879.922923,318.957958,0.362484
...,...,...,...,...,...
638,MAD-LHR,2023-12-24,1518.655656,857.993994,0.564969
639,MAD-LHR,2023-12-27,1658.294294,826.753754,0.498557
640,MAD-LHR,2023-12-29,590.689690,302.285285,0.511750
641,MAD-LHR,2023-12-30,1729.725726,649.066066,0.375242


In [60]:
# fake_data_agg.groupby('route').rolling(7)
fake_data_agg = fake_data_agg.sort_values(['route','date'])
fake_data_agg

,route,date,user_id,searches,device,revenue,revenue_imputed
0,BCN-LHR,2023-01-01,4,1402.945445,desktopdesktop,540.803303,1
1,BCN-LHR,2023-01-05,7,1334.598098,desktopdesktop,636.516517,0
2,BCN-LHR,2023-01-07,6,594.173173,desktop,239.276276,0
3,BCN-LHR,2023-01-08,10,1458.190190,desktopdesktop,648.921922,0
4,BCN-LHR,2023-01-10,2,879.922923,mobile,318.957958,0
...,...,...,...,...,...,...,...
638,MAD-LHR,2023-12-24,14,1518.655656,desktopmobile,857.993994,0
639,MAD-LHR,2023-12-27,17,1658.294294,desktopdesktopmobile,826.753754,0
640,MAD-LHR,2023-12-29,6,590.689690,mobile,302.285285,0
641,MAD-LHR,2023-12-30,21,1729.725726,mobiledesktopdesktop,649.066066,2


In [66]:
fake_data_agg['rolling_searches'] = (
    fake_data_agg.groupby('route')['total_searches']
    .transform(lambda x: x.rolling(7).mean())
)

fake_data_agg

,route,date,total_searches,total_revenue,avg_revenue_per_search,rolling_searches
0,BCN-LHR,2023-01-01,1402.945445,540.803303,0.385477,NaN
1,BCN-LHR,2023-01-05,1334.598098,636.516517,0.476935,NaN
2,BCN-LHR,2023-01-07,594.173173,239.276276,0.402705,NaN
3,BCN-LHR,2023-01-08,1458.190190,648.921922,0.445019,NaN
4,BCN-LHR,2023-01-10,879.922923,318.957958,0.362484,NaN
...,...,...,...,...,...,...
638,MAD-LHR,2023-12-24,1518.655656,857.993994,0.564969,1480.569141
639,MAD-LHR,2023-12-27,1658.294294,826.753754,0.498557,1277.651080
640,MAD-LHR,2023-12-29,590.689690,302.285285,0.511750,1309.584299
641,MAD-LHR,2023-12-30,1729.725726,649.066066,0.375242,1419.752610


In [67]:
fake_data_agg[fake_data_agg['route']=='MAD-LHR']

,route,date,total_searches,total_revenue,avg_revenue_per_search,rolling_searches
436,MAD-LHR,2023-01-02,605.494494,233.435435,0.385529,NaN
437,MAD-LHR,2023-01-04,554.906907,199.585586,0.359674,NaN
438,MAD-LHR,2023-01-05,339.414414,119.135135,0.351002,NaN
439,MAD-LHR,2023-01-07,443.538539,144.021021,0.324709,NaN
440,MAD-LHR,2023-01-08,1068.290290,499.411411,0.467487,NaN
...,...,...,...,...,...,...
638,MAD-LHR,2023-12-24,1518.655656,857.993994,0.564969,1480.569141
639,MAD-LHR,2023-12-27,1658.294294,826.753754,0.498557,1277.651080
640,MAD-LHR,2023-12-29,590.689690,302.285285,0.511750,1309.584299
641,MAD-LHR,2023-12-30,1729.725726,649.066066,0.375242,1419.752610
